In [2]:
import zipfile
import os

zip_path = "/content/Fake News Dataset.zip"
extract_path = "/content/Fake News Dataset"

# Extract if not already
if not os.path.exists(extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

print("Extraction done.")


Extraction done.


In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load your CSVs
df_real = pd.read_csv("/content/Fake News Dataset/LabeledAuthentic-7K.csv")
df_fake = pd.read_csv("/content/Fake News Dataset/LabeledFake-1K.csv")

df_fake_aug = pd.read_csv("/content/fake_train_augmented_FIXED (1).csv")



# Keep only content and label columns
df_real = df_real[['content', 'label', 'category']]
df_fake = df_fake[['content', 'label', 'category']]
df_fake_aug = df_fake_aug[['content', 'label', 'category']]

# Split each dataset individually: 70% train, 30% test
train_real, test_real = train_test_split(df_real, test_size=0.3, random_state=50, stratify=df_real['label'])

print("Train REAL category counts:")
print(train_real['category'].value_counts(), "\n")

train_fake, test_fake = train_test_split(df_fake, test_size=0.3, random_state=50, stratify=df_fake['label'])

train_fake = pd.concat([train_fake, df_fake_aug]).sample(frac=1, random_state=50)


print("Train FAKE category counts:")
print(train_fake['category'].value_counts(), "\n")


# Concatenate train splits and test splits
train_df = pd.concat([train_real, train_fake]).sample(frac=1, random_state=50).reset_index(drop=True)
test_df = pd.concat([test_real, test_fake]).sample(frac=1, random_state=50).reset_index(drop=True)




# Optional: check sizes and distribution
print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")
print(f"Train label distribution:\n{train_df['label'].value_counts()}")
print(f"Test label distribution:\n{test_df['label'].value_counts()}")


Train REAL category counts:
category
National         2431
Sports            619
International     613
Politics          273
Editorial         236
Entertainment     205
Miscellaneous     203
Crime             159
Finance            91
Education          73
Technology         71
Lifestyle          67
Name: count, dtype: int64 

Train FAKE category counts:
category
Miscellaneous     2791
Lifestyle          450
National           432
Entertainment      402
Politics           366
International      336
Sports             240
Crime              162
Technology         132
Education          132
Finance              6
0Miscellaneous       5
Name: count, dtype: int64 

Train size: 10495, Test size: 2551
Train label distribution:
label
0.0    5454
1.0    5041
Name: count, dtype: int64
Test label distribution:
label
1.0    2161
0.0     390
Name: count, dtype: int64


In [4]:
print("Rows in real CSV:", len(df_real))
print("Rows in fake CSV:", len(df_fake))
print("Total rows:", len(df_real) + len(df_fake))

Rows in real CSV: 7202
Rows in fake CSV: 1299
Total rows: 8501


In [ ]:
# Cell 2: Tokenization and Dataset
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

MODEL_NAME = "sagorsarker/bangla-bert-base"
MAX_LENGTH = 512
BATCH_SIZE = 8

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Create datasets
train_dataset = FakeNewsDataset(train_df['content'].tolist(), train_df['label'].tolist(), tokenizer, MAX_LENGTH)
test_dataset = FakeNewsDataset(test_df['content'].tolist(), test_df['label'].tolist(), tokenizer, MAX_LENGTH)

# Optional: create a dev set (10% of training data)
from sklearn.model_selection import train_test_split
train_texts, dev_texts, train_labels, dev_labels = train_test_split(
    train_df['content'].tolist(), train_df['label'].tolist(), test_size=0.1, random_state=50, stratify=train_df['label']
)
dev_dataset = FakeNewsDataset(dev_texts, dev_labels, tokenizer, MAX_LENGTH)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/491 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [ ]:
# Cell 3: Model & Training Setup
from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW   # <-- change here
import torch.nn.functional as F


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)
EPOCHS = 3


model.safetensors:   0%|          | 0.00/660M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at sagorsarker/bangla-bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Cell 4: Training Loop
from tqdm import tqdm

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    print(f"Epoch {epoch+1} | Avg Train Loss: {avg_train_loss:.4f}")

    # Evaluation on dev set
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in dev_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    print(f"Dev Accuracy: {correct/total:.4f}")


Training Epoch 1: 100%|██████████| 744/744 [09:43<00:00,  1.28it/s]


Epoch 1 | Avg Train Loss: 0.1954
Dev Accuracy: 0.9681


Training Epoch 2: 100%|██████████| 744/744 [09:50<00:00,  1.26it/s]


Epoch 2 | Avg Train Loss: 0.0916
Dev Accuracy: 0.9849


Training Epoch 3: 100%|██████████| 744/744 [09:50<00:00,  1.26it/s]


Epoch 3 | Avg Train Loss: 0.0382
Dev Accuracy: 0.9966


In [ ]:
# Cell 5: Test Evaluation
from sklearn.metrics import classification_report

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(all_labels, all_preds))


              precision    recall  f1-score   support

           0       0.91      0.82      0.86       390
           1       0.97      0.99      0.98      2161

    accuracy                           0.96      2551
   macro avg       0.94      0.90      0.92      2551
weighted avg       0.96      0.96      0.96      2551

